<img src="Bilder/ost_logo.png" width="240" align="right"/>
<div style="text-align: left"> <b> Machine Learning Projekt</b> <br> HS 2025 <br> <a href="mailto:ryan.mikus@ost.ch"> Ryan Mikus </a> und <a href="mailto:ilija.amato@ost.ch"> Ilija Amato </a> </div>

# 2. Datensammlung

## 2.1. Überblick

Für das Projekt müssen Daten beschaffen werden. Das Aktienuniversum ist der **S&P 500**. Dieses Aktienuniversum wurde gewählt, weil er breit, liquid und weil die Aktien alle auf dem selben Börsenmarkt sind (vereinfacht z.B. Unterschiede in Öffnungszeiten).

Die Datensammlung wird in zwei Schritten gemacht:

1. Abruf und Speicherung des Aktienuniversums (S&P-500-Ticker)
2. Abruf und Speicherung der Intraday-Kurszeitreihen

In [ ]:
# Zelle 1
from pairs_trading import data_io

DATE = "2026-01-05"
INTERVAL = "1m"
DATA_DIR = "Daten"

## 2.2. Auswahl des Aktienuniversums (S&P-500-Ticker)

Als Aktienuniversum verwenden wir den **S&P 500**, einen der wichtigsten US-Aktienindizes.  
Anstatt die Ticker manuell zu pflegen, werden sie automatisiert von Wikipedia geladen. Hierzu wird die Seite  
[*List of S&P 500 companies*](https://en.wikipedia.org/wiki/List_of_S%26P_500_companies) verwendet.

Mit `pandas.read_html` wird die erste Tabelle der Seite eingelesen, aus der die Spalte `Symbol` extrahiert und in eine Python-Liste umgewandelt wird.  
Da Yahoo Finance keine Punkte (`.`) in den Tickersymbolen akzeptiert (z. B. `BRK.B` → `BRK-B`), werden diese automatisch ersetzt.

Abschließend werden alle bereinigten Symbole in einer CSV-Datei gespeichert:

- **Datei:** `Daten/sp500_tickers.csv`  
- **Spalte:** `symbol`  
- Kein Index (für einfache Weiterverarbeitung)

Damit steht eine reproduzierbare und leicht aktualisierbare Liste der S&P-500-Ticker zur Verfügung.


In [10]:
# Zelle 2
universe = data_io.get_sp500_universe()

Lade S&P 500 Universe von Wikipedia …
Universe Größe: 503


## 2.3. Abruf der Intraday-Kursdaten über die Yahoo-Finance-API

Für die eigentlichen Kursdaten nutzen wir **1-Minuten-Intradaydaten** von **Yahoo Finance**.  
Da die Python-Bibliothek `yfinance` in der Projektumgebung nicht einsatzfähig war, erfolgt der Abruf direkt über die inoffizielle JSON-API von Yahoo Finance  
(`https://query1.finance.yahoo.com/v8/finance/chart/{ticker}`).

Dieser Ansatz ist in der Community gut dokumentiert, auch wenn es sich nicht um eine offiziell unterstützte Schnittstelle handelt.  
Ein Beispiel für eine Beschreibung der API ist hier zu finden:  
[Guide to Yahoo Finance API (Scrapfly)](https://scrapfly.io/blog/posts/guide-to-yahoo-finance-api)
Aufgrund einer Beschränkung der Yahoo-Finance-API stehen Kursdaten mit einer Frequenz von einer Minute nur für den aktuellen Handelstag beziehungsweise den zuletzt abgeschlossenen Börsentag (sofern kein neuer Handelstag begonnen hat) zur Verfügung.
Um dennoch einen fortlaufenden Datensatz aufzubauen, wird das Skript nach jedem Handelstag ausgeführt und speichert die neu verfügbaren Daten sukzessive ab.

### 2.3.1. Parameterdefinition

Zu Beginn werden die zentralen Parameter festgelegt:

- `DATE` → Datum des Abrufs  
- `INTERVAL` → Zeitintervall (z. B. 1m)  
- `RANGE` → Zeitraum (z. B. 1d)  
- `OUTPUT_FILE` → Pfad für die Zieldatei

Die zuvor gespeicherten Ticker werden eingelesen, und für jeden Ticker werden über die Funktion `fetch_yahoo_chart(...)` die Kurszeitreihen abgerufen.  
Die Funktion extrahiert die Zeitstempel und Kursdaten aus der JSON-Struktur und gibt sie als `pandas.DataFrame` zurück.

Fehlerhafte oder leere Antworten werden erkannt und übersprungen, um den Download-Prozess robust zu halten.


In [11]:
# Zelle 3
df = data_io.collect_sp500_intraday_data(
    universe=universe,
    date=DATE,
    interval=INTERVAL,
)

Sammle Intraday-Daten
Datum: 2026-01-02 | Interval: 1m
Progress: 100.00% |  503/ 503 | elapsed  244.0s | rows   196671
FERTIG
Zeilen: 196671
Ticker: 503
Zeitbereich: 2026-01-02 09:30:00-05:00 bis 2026-01-02 16:00:00-05:00


### 2.3.2. Schleife über alle S&P-500-Ticker und Speicherung

Für jeden Ticker im S&P 500 wird die Funktion `fetch_yahoo_chart(...)` aufgerufen.  
Die Daten werden gesammelt, in einer Liste zusammengeführt und schließlich als CSV-Datei gespeichert.

- Erfolgreich geladene Ticker werden zu `all_data` hinzugefügt  
- Fehlgeschlagene Ticker landen in der Liste `failed`  
- Nach jedem Request wird mit `time.sleep(0.1)` eine kleine Pause eingelegt, um das API-Rate-Limit zu respektieren

Nach Abschluss der Schleife:

- Alle DataFrames werden mit `pd.concat` zu einem Gesamt-Dataset verbunden  
- Das Ergebnis wird gespeichert als  
  **`Daten/sp500_1min_2024-11-07.csv`**

Falls keine Daten geladen wurden, wird eine entsprechende Fehlermeldung ausgegeben.  
Eine Liste der nicht verarbeiteten Ticker dient zur schnellen Fehlerdiagnose.


In [12]:
# Zelle 4
data_io.save_unprocessed_data(
    df,
    date=DATE,
    interval=INTERVAL,
    data_dir=DATA_DIR,
)


[Unprocessed Data (to be saved)]
Shape: (196671, 7)
Spalten: ['symbol', 'datetime', 'open', 'high', 'low', 'close', 'volume']

Head:
  symbol                  datetime        open        high         low  \
0    MMM 2026-01-02 09:30:00-05:00  160.100006  160.350006  160.000000   
1    MMM 2026-01-02 09:31:00-05:00  160.115906  160.289993  160.000000   
2    MMM 2026-01-02 09:32:00-05:00  159.960007  160.139999  159.919998   
3    MMM 2026-01-02 09:33:00-05:00  160.020004  160.020004  159.660004   
4    MMM 2026-01-02 09:34:00-05:00  159.845001  159.869995  159.740005   

        close   volume  
0  160.235107  32035.0  
1  160.000000   2758.0  
2  159.975006   3368.0  
3  159.865005   2057.0  
4  159.740005   1976.0  

Gespeichert: Daten\Rohdaten\sp500_1min_2026-01-02.csv


## 2.4. Einlesen und erste Plausibilitätsprüfung

Im letzten Schritt wird geprüft, ob die Datensammlung erfolgreich war.  
Dazu wird die erzeugte Datei `Daten/sp500_1min_2024-11-07.csv` wieder eingelesen und überprüft:

- Anzahl der Zeilen und Ticker  
- Zeitbereich der geladenen Daten (`datetime.min()` → `datetime.max()`)  
- Kontrolle, ob `datetime` korrekt als Datumsobjekt erkannt wurde  

Diese Prüfung bestätigt, dass die Daten konsistent und vollständig sind.  
Die so gewonnenen 1-Minuten-Kursdaten bilden die Grundlage für die anschließende **explorative Datenanalyse**, die im nächsten Kapitel behandelt wird:  
[03_Explorative_Datenanalyse](03_Explorative_Datenanalyse.html)


---

<div style="display: flex; justify-content: space-between; margin-top: 30px;">
  <a href="01_Einleitung.html">
    <button style="background-color:#007acc;color:white;border:none;padding:8px 12px;border-radius:6px;cursor:pointer;">
      Zurück
    </button>
  </a>
  <a href="03_Preprocessing.html">
    <button style="background-color:#007acc;color:white;border:none;padding:8px 12px;border-radius:6px;cursor:pointer;">
      Weiter
    </button>
  </a>
</div>
